# Training a Potential: how to setup data pipeline, model and trainer 

# Training a Nerual Network Potential

This section introduces how to use `MolPot` to train a nnp

## Preparing and loding data

Before we can start training neural networks with `MolPot`, we need to prepare our data. 

In [1]:
%load_ext autoreload
%autoreload 2

import molpot as mpot
import torch
from molpot import alias

/opt/conda/lib/python3.12/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [2]:
# 1. get rMD17 dataset
rmd17_ds = mpot.dataset.rMD17(
    molecule="aspirin",
    save_dir="data",
    device="cuda"
)
rmd17_ds.add_process(mpot.process.NeighborList(cutoff=5.0))

In [3]:
data_inspector = mpot.inspector.DataInspector(rmd17_ds)
data_inspector.summary()

number of data: 1000

                 dataset: rmd17-aspirin                 
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ label  ┃      type       ┃    unit    ┃   comment    ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ energy │ <class 'float'> │  kcal/mol  │ total energy │
│ forces │ <class 'float'> │ kcal/mol/A │  all forces  │
└────────┴─────────────────┴────────────┴──────────────┘

In [4]:
train_ds, valid_ds = torch.utils.data.random_split(rmd17_ds, [.95, .05])

train_dl = mpot.DataLoader(
    dataset=train_ds,
    batch_size=10,
    shuffle=False,
)
eval_dl = mpot.DataLoader(
    dataset=valid_ds,
    batch_size=10,
    shuffle=False,
)

## Setting up the model

In [ ]:
pinet = mpot.potential.nnp.PiNet(
    depth=5,
    basis_fn=mpot.potential.nnp.radial.GaussianRBF(10, 5.0),
    cutoff_fn=mpot.potential.nnp.cutoff.CosineCutoff(5.0),
    pi_nodes=[64, 64],
    ii_nodes=[64, 64, 64, 64],
    pp_nodes=[64, 64, 64, 64],
    activation=torch.nn.Tanh(),
    rank=1
)
e_readout = mpot.potential.nnp.readout.Atomwise(
    [64, 1],
    from_key=("pinet", "p1"),
    to_key=("predicts", "energy")
)
f_readout = mpot.potential.nnp.readout.DPairPot(
    fx_key=("predicts", "energy"),
    dx_key=("pairs", "dist"),
    to_key=("predicts", "forces"),
    create_graph=True
)
model = mpot.potential.PotentialSeq("pinet", pinet, e_readout, f_readout)

In [22]:
from ignite.metrics import MeanAbsoluteError, BatchWise
from pathlib import Path

loss_fn = mpot.loss.MultiTargetLoss(torch.nn.MSELoss(), [("forces", "forces", 1.0)])

trainer = mpot.PotentialTrainer(
    model,
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-4),
    loss_fn=loss_fn,
    device="cuda",
)
trainer.add_evaluator(eval_dl, no_grad=False)

def get_energy(data):
    return (data[0]["energy"], data[1]["energy"])

def get_forces(data):
    return (data[0]["forces"], data[1]["forces"])

trainer.add_metric(
    "e_mae",
    MeanAbsoluteError(output_transform=get_energy),
    target="all",
    usage=BatchWise(),
)
trainer.add_metric(
    "f_mae",
    MeanAbsoluteError(output_transform=get_forces),
    target="all",
    usage=BatchWise(),
)

trainer.attach_progressbar()
# trainer.attach_tensorboard(log_dir=Path("rmd17_log"))
from ignite.handlers import global_step_from_engine
from ignite.handlers.tensorboard_logger import TensorboardLogger
from ignite.engine.events import Events
tb_logger = TensorboardLogger(Path("rmd17_log"))
tb_logger.attach_output_handler(
    trainer.trainer,
    event_name=Events.ITERATION_COMPLETED,
    tag="trainer",
    output_transform=lambda x: {"loss": x[-1]},
    global_step_transform=global_step_from_engine(trainer.trainer),
)

tb_logger.attach_output_handler(
    trainer.trainer,
    event_name=Events.ITERATION_COMPLETED,
    tag="trainer",
    metric_names=["e_mae", "f_mae"],
    global_step_transform=global_step_from_engine(trainer.trainer),
)

tb_logger.attach_output_handler(
    trainer.evaluator,
    event_name=Events.EPOCH_COMPLETED,
    tag="evaluator",
    metric_names=["e_mae", "f_mae"],
    global_step_transform=global_step_from_engine(trainer.trainer),
)

state = trainer.run(train_dl, max_epochs=1000)

[1/95]   1%|1          [00:00<?]

[1/5]  20%|##         [00:00<?]

[1/95]   1%|1          [00:00<?]

[1/5]  20%|##         [00:00<?]

[1/95]   1%|1          [00:00<?]

...

[1/95]   1%|1          [00:00<?]

[1/5]  20%|##         [00:00<?]

[1/95]   1%|1          [00:00<?]

Engine run is terminating due to exception: 


KeyboardInterrupt: 